Compare and summarize all model performances by manually computing evaluation metrics, analyzing trade-offs in bias-variance, and justifying model selection based on statistical significance.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from scipy.stats import binomtest

df = pd.read_csv("heart_disease_train.csv")

features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
target = "target"

df = df[features + [target]].copy()

for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()

X = df[features].to_numpy(dtype=float)
y = df[target].to_numpy(dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 63
Testing samples: 16


In [2]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000)
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42
    ),
    "SVM": make_pipeline(
        StandardScaler(), SVC(probability=True, random_state=42)
    ),
    "KNN": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=5)
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained")

Logistic Regression trained
Decision Tree trained
Random Forest trained
SVM trained
KNN trained


c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

names = list(models.keys())
oof_prob = np.zeros((len(X_train), len(names)))
test_prob = np.zeros((len(X_test), len(names)))

for j, name in enumerate(names):
    for train_idx, val_idx in cv.split(X_train, y_train):
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000)
        ) if name == "Logistic Regression" else None

        # Clone the original model for each fold
        from sklearn.base import clone
        model = clone(models[name])
        model.fit(X_train[train_idx], y_train[train_idx])

        oof_prob[val_idx, j] = model.predict_proba(
            X_train[val_idx]
        )[:, 1]

    test_prob[:, j] = models[name].predict_proba(X_test)[:, 1]

# Manual hard voting
individual_test_predictions = np.column_stack([
    (test_prob[:, j] >= 0.5).astype(int)
    for j in range(len(names))
])

hard_vote = (
    individual_test_predictions.sum(axis=1) > len(names) / 2
).astype(int)

# Manual soft voting
soft_vote = (test_prob.mean(axis=1) >= 0.5).astype(int)

# Weighted soft voting using out-of-fold accuracy
oof_predictions = (oof_prob >= 0.5).astype(int)
model_scores = (oof_predictions == y_train[:, None]).mean(axis=0)

weights = model_scores / model_scores.sum()
weighted_vote = (
    np.average(test_prob, axis=1, weights=weights) >= 0.5
).astype(int)

# Manual stacking with a logistic regression meta-model
meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(oof_prob, y_train)
stacking_pred = meta_model.predict(test_prob)

print("Model weights:")
for name, weight in zip(names, weights):
    print(name, ":", round(weight, 3))

Model weights:
Logistic Regression : 0.201
Decision Tree : 0.193
Random Forest : 0.208
SVM : 0.201
KNN : 0.197


c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\

In [4]:
def manual_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return accuracy, precision, recall, f1

all_predictions = {}

for name, model in models.items():
    all_predictions[name] = model.predict(X_test)

all_predictions["Hard Voting"] = hard_vote
all_predictions["Soft Voting"] = soft_vote
all_predictions["Weighted Soft Voting"] = weighted_vote
all_predictions["Stacking"] = stacking_pred

results = []

for name, pred in all_predictions.items():
    acc, prec, rec, f1 = manual_metrics(y_test, pred)
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1
    })

results_df = pd.DataFrame(results).sort_values(
    "F1 Score", ascending=False
)

print(results_df.round(4).to_string(index=False))

               Model  Accuracy  Precision  Recall  F1 Score
Weighted Soft Voting    0.8125     0.8462  0.9167    0.8800
         Soft Voting    0.8125     0.8462  0.9167    0.8800
            Stacking    0.8125     0.8462  0.9167    0.8800
 Logistic Regression    0.7500     0.7857  0.9167    0.8462
       Random Forest    0.7500     0.7857  0.9167    0.8462
         Hard Voting    0.7500     0.7857  0.9167    0.8462
       Decision Tree    0.7500     0.8333  0.8333    0.8333
                 KNN    0.6875     0.7333  0.9167    0.8148
                 SVM    0.6875     0.7692  0.8333    0.8000


In [5]:
bias_variance = []

for name, model in models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_acc = np.mean(train_pred == y_train)
    test_acc = np.mean(test_pred == y_test)

    bias_variance.append({
        "Model": name,
        "Training Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Accuracy Gap": train_acc - test_acc
    })

bias_df = pd.DataFrame(bias_variance)
print(bias_df.round(4).to_string(index=False))

              Model  Training Accuracy  Test Accuracy  Accuracy Gap
Logistic Regression             0.8730         0.7500        0.1230
      Decision Tree             1.0000         0.7500        0.2500
      Random Forest             1.0000         0.7500        0.2500
                SVM             0.9524         0.6875        0.2649
                KNN             0.8254         0.6875        0.1379


In [6]:
# Select the top two models by F1 score
ranked_names = results_df["Model"].tolist()

model_a = ranked_names[0]
model_b = ranked_names[1]

pred_a = all_predictions[model_a]
pred_b = all_predictions[model_b]

a_correct = pred_a == y_test
b_correct = pred_b == y_test

# Cases where only one model is correct
only_a_correct = np.sum(a_correct & ~b_correct)
only_b_correct = np.sum(~a_correct & b_correct)

discordant = only_a_correct + only_b_correct

if discordant == 0:
    p_value = 1.0
else:
    p_value = binomtest(
        only_a_correct,
        n=discordant,
        p=0.5,
        alternative="two-sided"
    ).pvalue

print("Model A:", model_a)
print("Model B:", model_b)
print("Only Model A correct:", only_a_correct)
print("Only Model B correct:", only_b_correct)
print("McNemar exact p-value:", round(p_value, 4))

if p_value < 0.05:
    print("The difference is statistically significant at the 5% level.")
else:
    print("There is insufficient evidence of a significant difference.")

Model A: Weighted Soft Voting
Model B: Soft Voting
Only Model A correct: 0
Only Model B correct: 0
McNemar exact p-value: 1.0
There is insufficient evidence of a significant difference.


In [7]:
rng = np.random.default_rng(42)
n_bootstrap = 2000

for name, pred in all_predictions.items():
    correct = (pred == y_test).astype(float)

    bootstrap_scores = []

    for _ in range(n_bootstrap):
        sample = rng.choice(correct, size=len(correct), replace=True)
        bootstrap_scores.append(sample.mean())

    lower = np.percentile(bootstrap_scores, 2.5)
    upper = np.percentile(bootstrap_scores, 97.5)

    print(
        name,
        "| Accuracy:", round(correct.mean(), 3),
        "| 95% bootstrap CI:",
        (round(lower, 3), round(upper, 3))
    )

Logistic Regression | Accuracy: 0.75 | 95% bootstrap CI: (np.float64(0.562), np.float64(0.938))
Decision Tree | Accuracy: 0.75 | 95% bootstrap CI: (np.float64(0.5), np.float64(0.938))
Random Forest | Accuracy: 0.75 | 95% bootstrap CI: (np.float64(0.5), np.float64(0.938))
SVM | Accuracy: 0.688 | 95% bootstrap CI: (np.float64(0.438), np.float64(0.875))
KNN | Accuracy: 0.688 | 95% bootstrap CI: (np.float64(0.5), np.float64(0.875))
Hard Voting | Accuracy: 0.75 | 95% bootstrap CI: (np.float64(0.5), np.float64(0.938))
Soft Voting | Accuracy: 0.812 | 95% bootstrap CI: (np.float64(0.625), np.float64(1.0))
Weighted Soft Voting | Accuracy: 0.812 | 95% bootstrap CI: (np.float64(0.625), np.float64(1.0))
Stacking | Accuracy: 0.812 | 95% bootstrap CI: (np.float64(0.625), np.float64(1.0))
